# Demo notebook to create smaller, standardized and more usable .nc files.


In [1]:
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings("ignore")

import sys
import os
sys.path.append(os.path.abspath('../Pathfinder'))

from helper_functions import *
from radar_functions import *
from pathfinder import *
from attach_grounddata import *
from add_dataflashlog import *
from clicki_tool import *

%matplotlib inline

In [2]:
#TODO: translate internal layers into depth within snowpack [m]

### Load RADAR object

In [24]:
# datasetID = '24_04_2025_4'
# campaignID = '2025_St3TARTFO'
# RADAR2 = load_RADAR(radar_type='UWiBaSS',
#                    datasetID=datasetID,
#                    campaignID=campaignID,
#                    yaml_file="./campaigns.yaml"
#                    )


#for UWIBASS_PS files (or really anything non-SnowDrone setup)
path = '/Users/torka/AWI_2025/2025_10_09_15.pkl'
RADAR = load_RADAR(path=path)

Dataset  from campaign  is loaded from /Users/torka/AWI_2025/2025_10_09_15.pkl


In [12]:
%matplotlib qt
fig, ax = plt.subplots(figsize=(15, 5))

ax.imshow(RADAR.rx1_rpca, cmap='gray', aspect='auto')

ax.plot(RADAR.PF_bottom_interface)
ax.plot(RADAR.PF_top_interface)


In [26]:
(RADAR.slowtime[-1] - RADAR.slowtime[0]) / 60

np.float64(15.28580028216044)

In [23]:
fig, ax = plt.subplots(1,1,figsize=(15,5))

sns.kdeplot(np.diff(RADAR.slowtime))
sns.kdeplot(np.diff(RADAR2.timestamp))


<Axes: ylabel='Density'>

In [ ]:
RADAR.distance_travelled = np.cumsum(np.insert(np.sqrt(np.diff(RADAR.log_UTM_x)**2 + np.diff(RADAR.log_UTM_y)**2), 0, 0))

In [ ]:
RADAR.PF_snow_depth[~RADAR.altitude_mask] = np.nan
RADAR.PF_total_uncertainty[~RADAR.altitude_mask] = np.nan

### Write everything to a .nc file

In [ ]:
ds = xr.Dataset(
    data_vars=dict(
      radar_echogram = (("fasttime","time"), getattr(RADAR, 'rx_rpca')),
      snow_depth = (("time"), getattr(RADAR, 'PF_snow_depth')),
      snow_depth_uncertainty = (("time"), getattr(RADAR, 'PF_total_uncertainty')),
      air_snow_interface = (("time"), getattr(RADAR, 'PF_top_interface')),
      snow_ice_interface = (("time"), getattr(RADAR, 'PF_bottom_interface')),
      dielectric_constant = (("time"), getattr(RADAR, 'PF_snow_profile_eps_r')),
      dielectric_constant_uncertainty = (("time"), getattr(RADAR, 'PF_snow_profile_eps_r_uncertainty')),
      usage_mask = (("time"), getattr(RADAR, 'altitude_mask')),
      laser_altitude = (("time"), getattr(RADAR, 'CTUN_SAlt')),
      platform_yaw = (("time"), getattr(RADAR, 'yaw')),
      platform_roll = (("time"), getattr(RADAR, 'roll')),
      platform_pitch = (("time"), getattr(RADAR, 'pitch')),
      platform_pitch_compensated = (("time"), getattr(RADAR, 'compensated_pitch'))
    ),
    coords=dict(
      time = ("time", getattr(RADAR, 'timestamp')),
      fasttime = ("fasttime", getattr(RADAR, 'fasttime')),
      lat = ("time", getattr(RADAR, 'GPS_Lat')),
      lon = ("time", getattr(RADAR, 'GPS_Lng')),
      UTM_x = (("time"), getattr(RADAR, 'log_UTM_x')),
      UTM_y = (("time"), getattr(RADAR, 'log_UTM_y')),
      distance = (("time"), getattr(RADAR, 'distance_travelled'))
    )
)


In [ ]:
#coordinate descriptions
ds["time"].attrs.update({
    "standard_name": "time",
    "long_name": "Time of observation",
    "axis": "T",
    "calendar": "Seconds since 1970-01-01 00:00:00",
})
ds["lat"].attrs.update({
    "standard_name": "latitude",
    "long_name": "Latitude",
    "units": "degrees_north",
    "axis": "Y",
})
ds["lon"].attrs.update({
    "standard_name": "longitude",
    "long_name": "Longitude",
    "units": "degrees_east",
    "axis": "X",
})


ds["UTM_x"].attrs.update({
    "standard_name": "projection_x_coordinate",
    "long_name": "UTM easting",
    "units": "m",
    "grid_mapping": "utm_crs",
})
ds["UTM_y"].attrs.update({
    "standard_name": "projection_y_coordinate",
    "long_name": "UTM northing",
    "units": "m",
    "grid_mapping": "utm_crs",
})
ds["utm_crs"] = xr.Variable((), 0)  # scalar, no data
ds["utm_crs"].attrs.update({
    "grid_mapping_name": "transverse_mercator",
    "latitude_of_projection_origin": 0.0,
    "longitude_of_central_meridian": 15.0,
    "scale_factor_at_central_meridian": 0.9996,
    "false_easting": 500000.0,
    "false_northing": 0.0,
    "semi_major_axis": 6378137.0,
    "inverse_flattening": 298.257223563,
    "epsg_code": "EPSG:32633",
    "long_name": "UTM coordinate reference system",
})

ds["distance"].attrs.update({
    "long_name": "Distance travelled since takeoff",
    "units": "m",
})
#variable descriptions
ds["radar_echogram"].attrs.update({
    "long_name": "Radar echogram (fasttime vs. time)",
    "units": "V",  
    "description": "Radar echogram after RPCA has been applied to remove noise.",
})

ds["snow_depth"].attrs.update({
    "standard_name": "surface_snow_thickness",
    "long_name": "Snow thickness on surface",
    "units": "m",
    "description": "Snow thickness derived using Pathfinder method."
    
})

ds["snow_depth_uncertainty"].attrs.update({
    "long_name": "Uncertainty of snow thickness",
    "units": "m",
    "description": "Snow thickness uncertainty derived using Pathfinder method."
    
})
ds["air_snow_interface"].attrs.update({
    "units": "indices",
    "description": "Index of the air-snow interface in the radar echogram."
    
})
ds["snow_ice_interface"].attrs.update({
    "units": "indices",
    "description": "Index of the snow-ice interface in the radar echogram."

})
ds["dielectric_constant"].attrs.update({
    "long_name": "Relative permittivity of snow",
    "units": "1",
    "description": "Bulk estimate derived from nearby snow pits. Standard value: 1.5 if no snow pit data available.",
})

ds["dielectric_constant_uncertainty"].attrs.update({
    "long_name": "Uncertainty of relative permittivity of snow",
    "units": "1",
    "description": "Estimate derived using known error in-situ sampling and spatial variance. Standard value: 0.1 if no snow pit data available.",
    
})

ds["usage_mask"].attrs.update({
    "long_name": "Data usage mask",
    "units": "1",
    "flag_values": np.array([0, 1], dtype="i1"),
    "flag_meanings": "not_used used",
    "description": "Mask indicating which radar traces were used in Pathfinder."
})

ds["platform_yaw"].attrs.update({
    "standard_name": "platform_yaw",  
    "long_name": "Platform yaw angle",
    "units": "degree",
})
ds["platform_roll"].attrs.update({
    "standard_name": "platform_roll",  
    "long_name": "Platform roll angle",
    "units": "degree",
})
ds["platform_pitch"].attrs.update({
    "standard_name": "platform_pitch",  
    "long_name": "Platform pitch angle",
    "units": "degree",
})
ds["platform_pitch_compensated"].attrs.update({
    "long_name": "Pitch angle (compensated)",
    "units": "degree",
    "description": "Pitch angle accounting for the fixed +6 degree offset."
    
})

ds["laser_altitude"].attrs.update({
    "standard_name": "altitude",
    "long_name": "Platform altitude above surface, from laser altimeter",
    "units": "m",
    "positive": "up",
})

ds["fasttime"].attrs.update({
    "long_name": "Fast-time (two-way travel time from transmit)",
    "units": "ns", 
})

ds.attrs.update({
    "Conventions": "CF-1.10, ACDD-1.3",
    "title": "UAV-borne UWiBaSS snow survey",
    "summary": "Snow thickness, platform attitude, dielectric estimates, and radar echogram with georeferencing.",
    "institution": "NORCE Research AS",
    "source": "UAV-borne UWiBaSS + GNSS/INS",
    "keywords": "snow, radar, GPR, dielectric, platform attitude, UTM, latitude, longitude",
    "geospatial_lat_units": "degrees_north",
    "geospatial_lon_units": "degrees_east",
    "geospatial_vertical_units": "m",
    "time_coverage_start": str(pd.to_datetime(ds.time.min().values)),
    "time_coverage_end": str(pd.to_datetime(ds.time.max().values)),
    "standard_name_vocabulary": "CF Standard Name Table (current)",
})

ds.to_netcdf(os.path.join(RADAR.data_path, "Uwibass", datasetID, f"{datasetID}_outfile.nc"))